# Pretrained Centralized Shadow

Replace Qwen3-0.6B's lm head to Qwen8B's to obtain centralized shadow model for Qwen3-8B.

## 1. Initiliaze Centralized Shadow Model from Pretrained SLMs

In [2]:
base_model_name = 'Qwen/Qwen3-0.6B'
target_model_head = 'Qwen/Qwen3-8B'

In [ ]:
import copy
import torch
import torch.nn as nn
from shadow_peft import AutoModelForCausalLMWithHiddenProjection
from transformers import AutoModelForCausalLM, AutoTokenizer

/home/lxm/miniconda3/envs/py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Load the base model (Qwen3-0.6B) – we keep its backbone + original lm_head as reference
print(f"Loading base model: {base_model_name}")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
base_model.eval()

# Stash the original lm_head as the reference before we swap it out.
# shape: [vocab_size, shadow_hidden_size]  e.g. [151936, 1024] for Qwen3-0.6B
reference_lm_head = copy.deepcopy(base_model.lm_head)
for p in reference_lm_head.parameters():
    p.requires_grad = False

print(f"  Backbone hidden size : {base_model.config.hidden_size}")
print(f"  Reference lm_head   : {reference_lm_head.weight.shape}")

Loading base model: Qwen/Qwen3-0.6B


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 6882.87it/s]
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


  Backbone hidden size : 1024
  Reference lm_head   : torch.Size([151936, 1024])


In [5]:
# Load the target model (Qwen3-8B) only to extract its lm_head, then discard the rest.
print(f"Loading target model for lm_head: {target_model_head}")
target_model = AutoModelForCausalLM.from_pretrained(
    target_model_head,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
target_model.eval()

target_lm_head = copy.deepcopy(target_model.get_output_embeddings())
for p in target_lm_head.parameters():
    p.requires_grad = False

print(f"  Target lm_head      : {target_lm_head.weight.shape}")

# Free the full target model to save memory
del target_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("  Full target model deleted – memory freed.")

Loading target model for lm_head: Qwen/Qwen3-8B


Loading weights: 100%|██████████| 399/399 [00:00<00:00, 5839.73it/s]


  Target lm_head      : torch.Size([151936, 4096])
  Full target model deleted – memory freed.


In [6]:
# Derive projection dimensions
shadow_hidden_size = base_model.config.hidden_size   # e.g. 1024  (Qwen3-0.6B)
base_hidden_size   = target_lm_head.in_features      # e.g. 4096  (Qwen3-8B)

print(f"Projection: {shadow_hidden_size} -> {base_hidden_size}")

# A placeholder projection; its weights will be overwritten by the pseudoinverse initializer.
shadow_hidden_projection = nn.Linear(shadow_hidden_size, base_hidden_size, bias=False)

# Build the wrapped model.
# wrap() calls the pseudoinverse initialiser internally when init_optimal_projection=True:
#   solve  W_target @ W_proj.T ≈ W_reference
#   => W_proj.T = pinv(W_target) @ W_reference
wrapped_model = AutoModelForCausalLMWithHiddenProjection.wrap(
    shadow_model=base_model,
    shadow_hidden_projection=shadow_hidden_projection,
    lm_head=target_lm_head,
    init_optimal_projection=True,
    reference_lm_head=reference_lm_head,
)
print("Wrapped model created successfully.")
print(wrapped_model)

Projection: 1024 -> 4096


Initializing shadow_hidden_projection optimally via pseudoinverse (it will take a few minutes, please wait)...
  Reference lm_head shape: torch.Size([151936, 1024])
  Target lm_head shape: torch.Size([151936, 4096])
  Reconstruction error: 1.015625
  ✓ Optimal projection initialized
Wrapped model created successfully.
AutoModelForCausalLMWithHiddenProjection(
  (shadow_model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj

In [7]:
# Save the model (tokenizer comes from the base model)
output_dir = './initial_pretrained_shadows/qwen3-0.6b-with-8b-lm-head'

print(f"Saving model to: {output_dir}")
wrapped_model.save_pretrained(output_dir)

tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
tokenizer.save_pretrained(output_dir)

print("Done.")

Saving model to: ./initial_pretrained_shadows/qwen3-0.6b-with-8b-lm-head


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.57s/it]


Done.


### 1.1 Test

In [ ]:
# Sanity-check: reload from disk and run a tiny forward pass
reloaded = AutoModelForCausalLMWithHiddenProjection.from_pretrained(
    output_dir,
    torch_dtype=torch.bfloat16,
)
reloaded.eval()

tok = AutoTokenizer.from_pretrained(output_dir)
inputs = tok("Hello, world!", return_tensors="pt")
with torch.no_grad():
    out = reloaded(**inputs)

print("Logits shape:", out.logits.shape)   # should be [1, seq_len, vocab_size_of_8B]
print("Sanity check passed.")

Loading weights: 100%|██████████| 312/312 [00:00<00:00, 6392.07it/s]


Logits shape: torch.Size([1, 4, 151936])
Sanity check passed.


In [12]:
prompts = [
    "1+1=?",
    "Once upon a time in a land far away,",
    "The key difference between machine learning and deep learning is",
    "In Python, a list comprehension is",
]

reloaded.eval()
device = next(reloaded.parameters()).device

for prompt in prompts:
    inputs = tok(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = reloaded.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,          # greedy
            temperature=1.0,
            repetition_penalty=1.1,
        )
    # Decode only the newly generated tokens
    new_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    continuation = tok.decode(new_ids, skip_special_tokens=True)
    print(f"Prompt   : {prompt}")
    print(f"Generated: {continuation}")
    print("-" * 80)

Prompt   : 1+1=?
Generated:  What is the value of this expression?

Let's solve this problem step by step.

Step 1: Add 1 and 1.
Step 2: Calculate 1 + 1 = ?
Step 3: Answer: 2

So, 1 + 1 equals 2.

Answer: 2
--------------------------------------------------------------------------------
Prompt   : Once upon a time in a land far away,
Generated:  there was a village called Elan Valley. This village was known for its beautiful landscapes and vibrant community. One day, the villagers decided to celebrate their annual festival of Harvest Moon. The festival was celebrated on 21 days after midnight. On Day 1, they started harvesting crops. Each harvest season has 3
--------------------------------------------------------------------------------
Prompt   : The key difference between machine learning and deep learning is
Generated:  that deep learning uses multiple layers of neural networks (more complex structure) to perform more complex tasks. This makes deep learning capable of performing m

# 2. Next Step: Further Pretraining Centralized Shadow Model

refer to `experiment/run_lm_head_alignment.py`
